# 探索4,119対局：追加10特徴量
段位16以下/18以上、17除外、初期1500・同一時間帯・共通対局除外。探索末2024-07-01 02:00固定。追加8割合＋2量的。モデル学習なし。A1→A2→A3→A4→A5→A6。A4で親照合成功を確認してA5へ進む。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import os,sys,json,subprocess,urllib.request,hashlib
COMMIT='860605cf0f35f376be9dbd79763775c4300f11b4'
BASE=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{COMMIT}/research_v2/fast_release_20261126'
HOME=Path('/tmp')/('received_fast_'+COMMIT[:12])
SOURCE=HOME/'snapshot/research_v2'
manifest=json.loads(urllib.request.urlopen(BASE+'/source_manifest.json').read())
for relative,expected in manifest.items():
    p=HOME/'snapshot'/relative;p.parent.mkdir(parents=True,exist_ok=True)
    content=urllib.request.urlopen(BASE+'/snapshot/'+relative).read()
    if hashlib.sha256(content).hexdigest()!=expected: raise ValueError('固定ソース不一致: '+relative)
    if p.exists() and p.read_bytes()!=content: raise ValueError('ローカル固定版が変更されています: '+relative)
    if not p.exists():p.write_bytes(content)
for name in ['quality_supplement.py','test_supplement.py']:
    (HOME/name).write_bytes(urllib.request.urlopen(BASE+'/'+name).read())
DEPS=HOME/'deps';stamp=DEPS/'installed'
if not stamp.exists():
    subprocess.check_call([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(DEPS),'-r',str(SOURCE/'features_100/requirements.txt')])
    stamp.write_text(COMMIT)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[DEPS,HOME,SOURCE,SOURCE/'choices',SOURCE/'smoothing'])),PYTHONIOENCODING='utf-8',MPLBACKEND='Agg')
subprocess.check_call([sys.executable,'-m','unittest','discover','-s',str(SOURCE/'features_100/tests'),'-t',str(SOURCE),'-q'],env=ENV)
subprocess.check_call([sys.executable,'-m','unittest','test_supplement','-q'],env=ENV,cwd=HOME)
subprocess.run(['apt-get','install','-y','-qq','fonts-noto-cjk'],check=False,stdout=subprocess.DEVNULL)
print('固定ソース照合完了。実牌譜の検証完了ではありません。')


## A2 追加モジュール取得と合成テスト
固定高速版は変更しません。Z・全90候補の再計算を避けます。テスト成功は実牌譜検証を意味しません。

In [ ]:
ADD_COMMIT='70d6a64952610de562682c173321d45f641ad19d'
ADD=HOME/('additions_'+ADD_COMMIT[:12]);ADD.mkdir(exist_ok=True)
for name in ['core.py','workflow.py','test_additions.py','definitions.csv','README.md']:
    data=urllib.request.urlopen(f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{ADD_COMMIT}/research_v2/exploration_additions/{name}').read()
    target=ADD/name
    if target.exists() and target.read_bytes()!=data:raise ValueError('固定版不一致: '+name)
    if not target.exists():target.write_bytes(data)
ENV2=dict(ENV,PYTHONPATH=str(ADD)+os.pathsep+ENV['PYTHONPATH'])
subprocess.check_call([sys.executable,'-m','unittest','test_additions','-q'],env=ENV2,cwd=ADD)


## A3 対象・入力確認
旧全件カウンタの署名/ハッシュ/IDを照合。4119対局（3467/652）と合わなければ停止します。計数を合わせるために対象を削除しないでください。

In [ ]:
ROOT=Path('/content/drive/MyDrive/卒論')
CONFIG=dict(run_dir=str(ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'),baseline_dir=str(ROOT/'research_v2_features_100_fast/20261126T100602Z_994332bc'),player='しょぼんｎ',key='shobon',zip_override=None,output_root=str(ROOT/'research_v2_exploration_additions'))
CFG=ADD/'config.json';CFG.write_text(json.dumps(CONFIG,ensure_ascii=False),encoding='utf8')
def run(mode,extra=()):
    subprocess.check_call([sys.executable,str(ADD/'workflow.py'),mode,str(CFG),*extra],env=ENV2,cwd=ADD)
run('preview')


## A4 実牌譜パイロット・時間推定
各群3対局。親4項目の全計数/状態/値を既存全件結果と完全比較し不一致で停止。E5と同じIDとは限りません。pilotの新しい日時フォルダのaudit.csv、traces、complete.jsonを確認。未キャッシュ対局時間の4119対局単純外挿を出力。少数ログからの推定であり本番実測ではありません。

In [ ]:
run('pilot')


## A5 探索期間全件抽出・中断再開
A4の監査を確認後に実行。1セッションだけで実行。成功追加キャッシュを再利用、詳細トレースは保存しません。親カウンタ照合は全4119件で行います。中断後は同じ版・設定でA1〜A3、A5を実行。残存ロックは自動削除しません。

In [ ]:
RUN_FULL=False # A4確認後True
if RUN_FULL:run('full')
else:print('パイロット監査を確認してからRUN_FULL=Trueにしてください')


## A6 既存90＋追加10の品質比較
追加全件完了フォルダを指定。1対局・非重複10/20対局の統計、親子対応、時期別・群別相関。評価期間の数値は集計しません。量的に二項モデルを適用しません。分母0/unknownは欠損。

In [ ]:
FULL_DIR='' # research_v2_exploration_additions/full/日時 の実在フォルダ
if not FULL_DIR:raise ValueError('全件抽出の完了フォルダを指定してください')
run('quality',['--folder',FULL_DIR])


## 出力と限界
quality/日時 内のgroups.csv、comparison.csv、parent_child_same_windows.csv、periods.csv、correlations.csv、values.csv、report_ja.md。width=1は対局単位。10/20を混ぜない。群内分散とSMDだけで採否を決めず、機会数・欠損・時期差を確認。局内/時間依存の二項回帰は未実施。全期間の予測性能や因果効果ではありません。